In [14]:
# Brain Tumor Classification Demo with Gradio
# Complete implementation from model loading to Score-CAM visualization

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import cv2
import time

# Install Gradio
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "gradio", "-q"])
import gradio as gr

print('All imports successful')
print(f'Using device: {torch.device("cuda" if torch.cuda.is_available() else "cpu")}')

All imports successful
Using device: cpu


In [15]:
# Configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class_names = ['glioma', 'meningioma', 'notumor', 'pituitary']
num_classes = len(class_names)
INPUT_SIZE = 224
EMBED_DIM = 144
NUM_HEADS = 12
FF_DIM = 288
NUM_TRANSFORMER_BLOCKS = 2
DROPOUT = 0.2
GAMMA_VALUE = 1.5

print('Configuration loaded')

Configuration loaded


In [16]:
# Model architecture
class PatchEmbedding(nn.Module):
    def __init__(self, in_channels, embed_dim, patch_size=16):
        super(PatchEmbedding, self).__init__()
        self.patch_embed = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.norm = nn.LayerNorm(embed_dim)
    
    def forward(self, x):
        x = self.patch_embed(x)
        B, C, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)
        x = self.norm(x)
        return x

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super(MultiHeadSelfAttention, self).__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        assert self.head_dim * num_heads == embed_dim
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.proj = nn.Linear(embed_dim, embed_dim)
        
    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        attn = attn.softmax(dim=-1)
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        return x

class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super(TransformerEncoderBlock, self).__init__()
        self.attention = MultiHeadSelfAttention(embed_dim, num_heads)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.dropout1 = nn.Dropout(dropout)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, embed_dim),
            nn.Dropout(dropout)
        )
        self.norm2 = nn.LayerNorm(embed_dim)
        
    def forward(self, x):
        attn_output = self.attention(x)
        x = x + self.dropout1(attn_output)
        x = self.norm1(x)
        ffn_output = self.ffn(x)
        x = x + ffn_output
        x = self.norm2(x)
        return x

class MobileNetViT(nn.Module):
    def __init__(self, num_classes, embed_dim=144, num_heads=12, ff_dim=288, num_transformer_blocks=2, dropout=0.1):
        super(MobileNetViT, self).__init__()
        mobilenet_full = models.mobilenet_v2(weights=None)
        self.mobilenet = mobilenet_full.features
        self.adaptive_pool = nn.AdaptiveAvgPool2d((7, 7))
        mobilenet_out_channels = 1280
        self.patch_embedding = PatchEmbedding(in_channels=mobilenet_out_channels, embed_dim=embed_dim, patch_size=1)
        num_patches = 49
        self.pos_embedding = nn.Parameter(torch.randn(1, num_patches, embed_dim))
        self.transformer_blocks = nn.ModuleList([
            TransformerEncoderBlock(embed_dim, num_heads, ff_dim, dropout)
            for _ in range(num_transformer_blocks)
        ])
        self.local_conv = nn.Sequential(
            nn.Conv2d(mobilenet_out_channels, embed_dim, kernel_size=1),
            nn.BatchNorm2d(embed_dim),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )
        self.fusion = nn.Linear(embed_dim * 2, embed_dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )
        
    def forward(self, x):
        mobilenet_features = self.mobilenet(x)
        mobilenet_features = self.adaptive_pool(mobilenet_features)
        tokens = self.patch_embedding(mobilenet_features)
        tokens = tokens + self.pos_embedding
        num_tokens = tokens.shape[1]
        split_point = num_tokens // 2
        q1_tokens = tokens[:, :split_point, :]
        q2_tokens = tokens[:, split_point:, :]
        for transformer_block in self.transformer_blocks:
            q2_tokens = transformer_block(q2_tokens)
        global_features = q2_tokens.mean(dim=1)
        local_features = self.local_conv(mobilenet_features)
        local_features = local_features.flatten(1)
        fused_features = torch.cat([local_features, global_features], dim=1)
        fused_features = self.fusion(fused_features)
        fused_features = self.dropout(fused_features)
        output = self.classifier(fused_features)
        return output

print('Model architecture defined')

Model architecture defined


In [17]:
# Load model
print('Loading MobileNetViT model...')

mobilenetvit_model = MobileNetViT(
    num_classes=num_classes,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    ff_dim=FF_DIM,
    num_transformer_blocks=NUM_TRANSFORMER_BLOCKS,
    dropout=DROPOUT
)

mobilenetvit_checkpoint = torch.load(
    '/kaggle/input/datasets/rahmymuhammadh/brain-tumor-models/mobilenetvit_brain_tumor_model.pth', 
    map_location=device,
    weights_only=False
)

mobilenetvit_model.load_state_dict(mobilenetvit_checkpoint['model_state_dict'], strict=False)
mobilenetvit_model = mobilenetvit_model.to(device)
mobilenetvit_model.eval()

print('Model loaded successfully')
print(f'Total parameters: {sum(p.numel() for p in mobilenetvit_model.parameters()):,}')

Loading MobileNetViT model...
Model loaded successfully
Total parameters: 2,986,532


In [18]:
# Preprocessing functions
def apply_gamma_correction(image, gamma_value=1.5):
    normalized = image / 255.0
    gamma_corrected = np.power(normalized, gamma_value)
    corrected = (gamma_corrected * 255).astype(np.uint8)
    return corrected

def preprocess_image(image_pil, gamma_value=1.5):
    if image_pil.mode != 'RGB':
        image_pil = image_pil.convert('RGB')
    
    image_pil_resized = image_pil.resize((INPUT_SIZE, INPUT_SIZE), Image.LANCZOS)
    image_np = np.array(image_pil_resized)
    image_gamma = apply_gamma_correction(image_np, gamma_value)
    image_gamma_pil = Image.fromarray(image_gamma)
    image_float = image_gamma.astype(np.float32) / 255.0
    image_tensor = torch.from_numpy(image_float).permute(2, 0, 1)
    
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    image_tensor = (image_tensor - mean) / std
    image_tensor = image_tensor.unsqueeze(0)
    
    return image_gamma_pil, image_tensor

print('Preprocessing functions defined')

Preprocessing functions defined


In [19]:
# Optimized Score-CAM
class OptimizedScoreCAM:
    def __init__(self, model, target_layer, top_k=64):
        self.model = model
        self.target_layer = target_layer
        self.top_k = top_k
        self.activations = None
        self.handle = self.target_layer.register_forward_hook(self.save_activation)
    
    def save_activation(self, module, input, output):
        self.activations = output.detach()
    
    def generate_cam(self, input_image, target_class=None, batch_size=128):
        self.model.eval()
        
        with torch.no_grad():
            base_output = self.model(input_image)
            if target_class is None:
                target_class = base_output.argmax(dim=1).item()
        
        with torch.no_grad():
            _ = self.model(input_image)
        
        activations = self.activations[0]
        num_channels = activations.shape[0]
        
        if num_channels > self.top_k:
            channel_importance = activations.abs().mean(dim=(1, 2))
            top_indices = torch.topk(channel_importance, k=self.top_k).indices
            activations = activations[top_indices]
            num_channels = self.top_k
        
        activations_upsampled = F.interpolate(
            activations.unsqueeze(0),
            size=input_image.shape[2:],
            mode='bilinear',
            align_corners=False
        )[0]
        
        activations_normalized = []
        for i in range(num_channels):
            act = activations_upsampled[i]
            act_min = act.min()
            act_max = act.max()
            if act_max > act_min:
                act_norm = (act - act_min) / (act_max - act_min)
            else:
                act_norm = torch.zeros_like(act)
            activations_normalized.append(act_norm)
        
        activations_normalized = torch.stack(activations_normalized)
        
        weights = []
        
        for start_idx in range(0, num_channels, batch_size):
            end_idx = min(start_idx + batch_size, num_channels)
            batch_acts = activations_normalized[start_idx:end_idx]
            
            batch_masked_inputs = []
            for act in batch_acts:
                act_expanded = act.unsqueeze(0).expand(3, -1, -1)
                masked_input = input_image[0] * act_expanded
                batch_masked_inputs.append(masked_input)
            
            batch_masked_inputs = torch.stack(batch_masked_inputs)
            
            with torch.no_grad():
                batch_outputs = self.model(batch_masked_inputs)
                batch_scores = batch_outputs[:, target_class]
            
            weights.extend(batch_scores.cpu().numpy().tolist())
        
        weights = torch.tensor(weights, device=activations.device)
        weights = F.relu(weights)
        
        if weights.sum() > 0:
            weights = weights / weights.sum()
        
        cam = torch.zeros(activations.shape[1:], dtype=torch.float32, device=activations.device)
        for i in range(num_channels):
            cam += weights[i] * activations[i]
        
        cam = F.relu(cam)
        if cam.max() > 0:
            cam = cam / cam.max()
        
        return cam.cpu().numpy(), target_class
    
    def remove_hook(self):
        self.handle.remove()

print('Optimized Score-CAM defined')

Optimized Score-CAM defined


In [20]:
# Initialize Score-CAM
target_layer = mobilenetvit_model.mobilenet[-3]
scorecam = OptimizedScoreCAM(mobilenetvit_model, target_layer, top_k=64)

print('Score-CAM initialized')

Score-CAM initialized


In [21]:
# Visualization helper function
def apply_colormap_on_image(org_img, activation_map, colormap=cv2.COLORMAP_JET):
    heatmap = cv2.resize(activation_map, (org_img.shape[1], org_img.shape[0]))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, colormap)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    superimposed_img = heatmap * 0.4 + org_img * 0.6
    superimposed_img = np.clip(superimposed_img, 0, 255).astype(np.uint8)
    return heatmap, superimposed_img

print('Visualization helper defined')

Visualization helper defined


In [22]:
# Main prediction function for Gradio
def predict_and_visualize(image):
    """
    Complete pipeline: preprocess -> predict -> Score-CAM -> visualize
    """
    if image is None:
        return None, None, None, "Please upload an image first"
    
    # Convert to PIL if needed
    if isinstance(image, np.ndarray):
        image_pil = Image.fromarray(image.astype('uint8'))
    else:
        image_pil = image
    
    # Start timing
    start_time = time.time()
    
    # Preprocess
    gamma_img, img_tensor = preprocess_image(image_pil, gamma_value=GAMMA_VALUE)
    img_tensor = img_tensor.to(device)
    
    # Model inference
    inference_start = time.time()
    with torch.no_grad():
        outputs = mobilenetvit_model(img_tensor)
        probabilities = torch.softmax(outputs, dim=1)
        confidence, predicted_class = torch.max(probabilities, 1)
        
        predicted_label = class_names[predicted_class.item()]
        confidence_score = confidence.item() * 100
        all_probs = probabilities.cpu().numpy()[0] * 100
    inference_time = (time.time() - inference_start) * 1000
    
    # Generate Score-CAM
    scorecam_start = time.time()
    cam, _ = scorecam.generate_cam(img_tensor, target_class=predicted_class.item(), batch_size=128)
    scorecam_time = (time.time() - scorecam_start) * 1000
    
    total_time = time.time() - start_time
    
    # Prepare visualizations
    gamma_img_np = np.array(gamma_img)
    heatmap, overlay = apply_colormap_on_image(gamma_img_np, cam)
    
    # Format class name
    display_label = predicted_label.upper()
    if predicted_label == 'notumor':
        display_label = 'NO TUMOR'
    
    # Create results markdown
    results_text = f"""
## 🎯 Prediction Results

### **Predicted Class: {display_label}**
### **Confidence: {confidence_score:.2f}%**

---

### 📊 Class Probabilities:

| Class | Probability | Bar |
|-------|-------------|-----|
| **Glioma** | {all_probs[0]:.2f}% | {'█' * int(all_probs[0]/5)}{'░' * (20-int(all_probs[0]/5))} |
| **Meningioma** | {all_probs[1]:.2f}% | {'█' * int(all_probs[1]/5)}{'░' * (20-int(all_probs[1]/5))} |
| **No Tumor** | {all_probs[2]:.2f}% | {'█' * int(all_probs[2]/5)}{'░' * (20-int(all_probs[2]/5))} |
| **Pituitary** | {all_probs[3]:.2f}% | {'█' * int(all_probs[3]/5)}{'░' * (20-int(all_probs[3]/5))} |

---

### ⏱️ Performance Metrics:

- **Model Inference:** {inference_time:.1f} ms
- **Score-CAM Generation:** {scorecam_time:.1f} ms  
- **Total Processing Time:** {total_time:.2f} seconds

---

### 🔬 Model Information:

- **Architecture:** MobileNetViT (Hybrid CNN + Transformer)
- **Parameters:** 2.99 Million
- **Test Accuracy:** 99.16%
- **Explainability Method:** Optimized Score-CAM (Top-64 channels)
- **Target Layer:** MobileNetV2 Layer -3

---

### 💡 Interpretation:

Red regions in the heatmap indicate areas that **strongly influenced** the classification decision.  
The overlay combines the original MRI with the attention heatmap for clinical interpretation.
"""
    
    return gamma_img_np, heatmap, overlay, results_text

print('Main prediction function defined')

Main prediction function defined


In [23]:
# Create Gradio Interface - Clean and Professional
print('Creating Gradio interface...')

custom_css = """
.gradio-container {
    font-family: 'Arial', sans-serif;
    max-width: 1400px;
    margin: auto;
}
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Soft(), title="Brain Tumor Classification") as demo:
    
    gr.Markdown("# 🧠 Brain Tumor Classification with Explainability")
    gr.Markdown("**MobileNetViT | 99.16% Accuracy | 2.99M Parameters**")
    
    with gr.Row():
        with gr.Column(scale=1):
            input_image = gr.Image(type="pil", label="Upload Brain MRI Scan")
            analyze_btn = gr.Button("Analyze MRI", variant="primary", size="lg")
        
        with gr.Column(scale=2):
            with gr.Row():
                preprocessed_output = gr.Image(label="Preprocessed")
                heatmap_output = gr.Image(label="Heatmap")
                overlay_output = gr.Image(label="Overlay")
            
            results_output = gr.Markdown()
    
    analyze_btn.click(
        fn=predict_and_visualize,
        inputs=input_image,
        outputs=[preprocessed_output, heatmap_output, overlay_output, results_output]
    )

print('Gradio interface created')

Creating Gradio interface...


/tmp/ipykernel_55/1764164790.py:12: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Soft(), title="Brain Tumor Classification") as demo:
/tmp/ipykernel_55/1764164790.py:12: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Soft(), title="Brain Tumor Classification") as demo:


Gradio interface created


In [24]:
# Launch demo
print('Launching Gradio demo...')
print('A shareable link will be generated for your viva')

demo.launch(share=True, debug=False)

Launching Gradio demo...
A shareable link will be generated for your viva
* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://48cd7e482ac4bf6fb7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
